In [ ]:
from pathlib import Path

from weavehr_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
DATASET = "mimic-iv"
WEAVEHR_CONCEPT_VERSION = "1.0.0"
# WeavEHR dataset version; None resolves it and fails if several are present.
DATASET_VERSION = None

# Shared output directory for this project.
# The R/RICU scripts use the same directory by default.
OUTPUT_ROOT = Path.home() / "output" / "weavehr_yaib"

# Required: the WeavEHR concept directory, e.g. <project>/workspace/concept.
CONCEPT_ROOT = None

# Optional: override these paths if your local layout differs from the defaults.
# You can also set the corresponding environment variables:
#   WEAVEHR_YAIB_ICUSTAYS_CSV
#   WEAVEHR_YAIB_RICU_CONCEPT_DICT
ICUSTAYS_CSV = None
RICU_CONCEPT_DICT = None

# ---------------------------------------------------------------------------
# 1) Export all available ICU hours for downstream ML/training.
# ---------------------------------------------------------------------------
all_hours = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    version=WEAVEHR_CONCEPT_VERSION,
    max_hours=None,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote all-hours WeavEHR YAIB-wide parquet to: {all_hours.output_path}")
all_hours.summary


In [ ]:
from weavehr_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# 2) Export the first week only for R/RICU validation.
#
# This follows the RICU/YAIB convention used in the archive comparison:
# time = 0, 1, ..., 7 * 24, i.e. max time is inclusive.
# ---------------------------------------------------------------------------
MAX_HOURS = 7 * 24

one_week = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    version=WEAVEHR_CONCEPT_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    missing_concepts="fail",
)

print(f"Wrote one-week WeavEHR YAIB-wide parquet to: {one_week.output_path}")
one_week.summary


## Optional R/RICU reference export

Run the R scripts before executing the comparison cell if you want to compare the WeavEHR YAIB-wide parquet against R/RICU.

```bash
RICU_OUT_DIR="$HOME/output/weavehr_yaib" Rscript scripts/export_ricu_dynamic_vars.R
RICU_OUT_DIR="$HOME/output/weavehr_yaib" Rscript scripts/export_ricu_stay_windows.R
```

The comparison cell expects these files in `OUTPUT_ROOT`:

```text
ricu_dynamic_vars_miiv.parquet
ricu_stay_windows_miiv.parquet
```

The comparison uses the same window logic as the working archive notebook: RICU stay windows are converted to integer hours, capped to `MAX_HOURS`, and RICU dynamic rows are filtered to `start <= time <= end`.

`all_hours.output_path` is intended for downstream ML/training. `one_week.output_path` is only the bounded validation export.


In [ ]:
from weavehr_yaib import (
    compare_weavehr_wide_to_ricu_for_dataset,
    display_comparison_overview,
)

# Compare the one-week WeavEHR parquet against R/RICU.
# Expected valid mismatch pattern for the known issue: 14 window/stay differences.
comparison = compare_weavehr_wide_to_ricu_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    weavehr_wide_path=one_week.output_path,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)

print(f"Wrote comparison reports to: {comparison.reports_dir}")
display_comparison_overview(comparison)
